# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

c:\Users\manke\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
price_data = glob(os.path.join(os.getenv('PRICE_DATA')))

price_data

['../../05_src/data/prices/']

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

#ELT: Load
price_files = glob(os.path.join(os.getenv('PRICE_DATA'), '**/*.parquet'),recursive=True)
price_dd = dd.read_parquet(price_files).set_index('ticker')

In [7]:
price_dd.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year'],
      dtype='object')

In [ ]:
#ELT: Transform
dd_feat_price = price_dd.groupby('ticker',group_keys=False).apply(
    lambda lag: lag.assign(
        Close_lag_1 = lag['Close'].shift(1),
        Adj_Close_lag_1 = lag['Adj Close'].shift(1)
        )
    )


C:\Users\manke\AppData\Local\Temp\ipykernel_11636\104701967.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat_price = price_dd.groupby('ticker',group_keys=False).apply(


In [19]:
dd_feat_price = dd_feat_price.assign(
    returns = lambda x: x['Close']/x['Close_lag_1']-1,
    hi_lo_range = lambda x: x['High'] - x['Low']
    )

In [35]:
dd_feat_price.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-06-26,4.04,4.19,3.86,4.00,4.000000,515300.0,ZIXI.csv,2003,4.04,4.040000,-0.009901,0.33
ZIXI,2003-06-27,4.00,4.05,3.79,3.85,3.850000,162400.0,ZIXI.csv,2003,4.00,4.000000,-0.037500,0.26
ZIXI,2003-06-30,3.84,4.00,3.72,3.77,3.770000,119900.0,ZIXI.csv,2003,3.85,3.850000,-0.020779,0.28


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.
import pandas as pd

price_features_df = pd.DataFrame(dd_feat_price.compute())
price_features_df

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-06-26,4.04,4.19,3.86,4.00,4.000000,515300.0,ZIXI.csv,2003,4.04,4.040000,-0.009901,0.33
ZIXI,2003-06-27,4.00,4.05,3.79,3.85,3.850000,162400.0,ZIXI.csv,2003,4.00,4.000000,-0.037500,0.26
ZIXI,2003-06-30,3.84,4.00,3.72,3.77,3.770000,119900.0,ZIXI.csv,2003,3.85,3.850000,-0.020779,0.28


In [33]:
#Rolling Avg
price_features_df['10_day_avg'] = price_features_df['returns'].rolling(10).mean()

In [ ]:
price_features_df.loc[['ALDX']].head(11) #checking for correctness

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,10_day_avg
ticker,,,,,,,,,,,,,,
ALDX,2014-05-02,7.500,7.690,6.010,7.20,7.20,579900.0,ALDX.csv,2014,NaN,NaN,NaN,1.680,NaN
ALDX,2014-05-05,6.800,7.000,6.300,6.70,6.70,41100.0,ALDX.csv,2014,7.20,7.20,-0.069444,0.700,NaN
ALDX,2014-05-06,6.650,6.850,6.100,6.10,6.10,24900.0,ALDX.csv,2014,6.70,6.70,-0.089552,0.750,NaN
ALDX,2014-05-07,6.120,6.240,6.050,6.05,6.05,12700.0,ALDX.csv,2014,6.10,6.10,-0.008197,0.190,NaN
ALDX,2014-05-08,6.500,6.990,6.490,6.99,6.99,24200.0,ALDX.csv,2014,6.05,6.05,0.155372,0.500,NaN
ALDX,2014-05-09,6.980,6.980,6.550,6.55,6.55,1300.0,ALDX.csv,2014,6.99,6.99,-0.062947,0.430,NaN
ALDX,2014-05-12,6.980,6.980,6.700,6.93,6.93,3100.0,ALDX.csv,2014,6.55,6.55,0.058015,0.280,NaN
ALDX,2014-05-13,6.500,6.980,6.500,6.98,6.98,1400.0,ALDX.csv,2014,6.93,6.93,0.007215,0.480,NaN
ALDX,2014-05-14,6.954,6.954,6.520,6.52,6.52,200.0,ALDX.csv,2014,6.98,6.98,-0.065903,0.434,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it was not neccesary, but pandas has built-in methods for data manipulation and aggregation tasks that make it less computation and time-intensive, especially when it comes to inserting new columns

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.